# Instance Segmentation Benchmark on Kaggle 2x NVIDIA T4 GPU (Large Models)

Notebook ini melatih, mengevaluasi, mencatat benchmark GPU, serta mengekspor 3 arsitektur instance segmentation varian **Large** pada dataset **`panoptic-indoor-segmentation.v1i.yolov8-obb`**:
1. **Mask2Former Large** (`facebook/mask2former-swin-large-coco-instance`)
2. **Mask R-CNN Large** (`ResNet-101-FPN` + COCO-pretrained RPN/RoIHeads)
3. **YOLOv26-seg Large** (`yolo26l-seg.pt`)


In [ ]:
import os
import sys
import subprocess

INPUT_DIR = "/kaggle/input/datasets/melekbengi/data-viskom"
DATA_SUBDIR = "/kaggle/input/datasets/melekbengi/data-viskom/data viskom"
DATA_YAML_PATH = "/kaggle/input/datasets/melekbengi/data-viskom/data.yaml"
WORKING_DIR = "/kaggle/working/"

if not os.path.exists(WORKING_DIR):
    try:
        os.makedirs(WORKING_DIR, exist_ok=True)
    except OSError:
        WORKING_DIR = os.path.abspath("./working") + "/"
        os.makedirs(WORKING_DIR, exist_ok=True)

COCO_DIR = os.path.join(WORKING_DIR, "coco_annotations")
YOLO_SEG_DIR = os.path.join(WORKING_DIR, "yolo_dataset")
CHECKPOINT_DIR = os.path.join(WORKING_DIR, "checkpoints")
LOG_DIR = os.path.join(WORKING_DIR, "logs")
TFLITE_EXPORT_DIR = os.path.join(WORKING_DIR, "exports_tflite")
BENCHMARK_JSON_PATH = os.path.join(WORKING_DIR, "gpu_benchmark.json")
BENCHMARK_CSV_PATH = os.path.join(WORKING_DIR, "gpu_benchmark.csv")
LOGBOOK_PATH = os.path.join(WORKING_DIR, "experiment_logbook.md")

for folder in [COCO_DIR, YOLO_SEG_DIR, CHECKPOINT_DIR, LOG_DIR, TFLITE_EXPORT_DIR]:
    os.makedirs(folder, exist_ok=True)

IMG_SIZE = 448
EPOCHS = 15
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 0
SEED = 42
LR_MASK2FORMER = 2e-4
LR_MASKRCNN = 1.5e-4

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("INPUT_DIR      :", INPUT_DIR)
print("DATA_SUBDIR    :", DATA_SUBDIR)
print("DATA_YAML_PATH :", DATA_YAML_PATH)
print("WORKING_DIR    :", WORKING_DIR)


## 1. Instalasi Dependensi, Import Library & Setup Hardware Multi-GPU (2x T4)


In [ ]:
import importlib.metadata
import importlib.util
import numpy as np

constraint_lines = [f"numpy=={np.__version__}"]
for dist_name in ["scipy", "torch", "torchvision", "tensorflow", "albumentations"]:
    try:
        constraint_lines.append(f"{dist_name}=={importlib.metadata.version(dist_name)}")
    except importlib.metadata.PackageNotFoundError:
        pass

constraint_path = os.path.join(WORKING_DIR, "pip_constraints.txt")
with open(constraint_path, "w", encoding="utf-8") as cf:
    cf.write("\n".join(constraint_lines) + "\n")

required_packages = [
    ("ultralytics", "ultralytics>=8.3.0"),
    ("cloudpickle", "cloudpickle>=3.1.1"),
    ("pycocotools", "pycocotools>=2.0.7"),
    ("onnx", "onnx>=1.15.0"),
    ("onnxruntime", "onnxruntime>=1.17.0"),
    ("onnx2tf", "onnx2tf>=1.20.0"),
    ("onnx_graphsurgeon", "onnx-graphsurgeon"),
    ("transformers", "transformers>=4.40.0"),
    ("accelerate", "accelerate>=0.28.0"),
    ("timm", "timm>=0.9.16"),
    ("albumentations", "albumentations>=1.4.0"),
    ("tensorboard", "tensorboard>=2.15.0"),
    ("tensorflow", "tensorflow>=2.15.0"),
    ("yaml", "pyyaml>=6.0"),
]

missing_packages = [pkg for mod, pkg in required_packages if importlib.util.find_spec(mod) is None]
if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-c", constraint_path, *missing_packages],
        check=False,
    )

import contextlib
import copy
import csv
import gc
import inspect
import json
import math
import random
import shutil
import threading
import time
import types
import warnings
from pathlib import Path

try:
    import albumentations as A
except AttributeError as exc:
    if "numpy.ufunc" in str(exc) or "__module__" in str(exc):
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", f"numpy=={np.__version__}"],
            check=False,
        )
        raise RuntimeError(
            "Konflik versi numpy di disk dan memori terdeteksi akibat instalasi sebelumnya. "
            "Versi numpy telah dikembalikan ke " + np.__version__ + ". Silakan klik 'Restart Session' di Kaggle lalu jalankan ulang dari sel pertama."
        ) from exc
    raise

import cv2
import pandas as pd
import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import yaml
from PIL import Image
from pycocotools import mask as mask_util
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchvision.models import ResNet101_Weights
from torchvision.models.detection import MaskRCNN, maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from tqdm.auto import tqdm
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor
from transformers.models.mask2former.modeling_mask2former import Mask2FormerPixelDecoderEncoderOnly
from ultralytics import YOLO

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS >= 2:
    GPU_IDS = [0, 1]
    PRIMARY_DEVICE = torch.device("cuda:0")
elif NUM_GPUS == 1:
    GPU_IDS = [0]
    PRIMARY_DEVICE = torch.device("cuda:0")
else:
    GPU_IDS = []
    PRIMARY_DEVICE = torch.device("cpu")

USE_AMP = torch.cuda.is_available()
if USE_AMP:
    torch.backends.cudnn.benchmark = True

print(f"NumPy          : {np.__version__}")
print(f"CUDA available : {torch.cuda.is_available()} | GPU count : {NUM_GPUS} | GPU_IDS : {GPU_IDS}")
for gid in GPU_IDS:
    props = torch.cuda.get_device_properties(gid)
    print(f"GPU {gid}          : {props.name} ({props.total_memory / (1024 ** 3):.2f} GB VRAM)")


## 2. Resolusi Dataset & Konversi Format YOLOv8-OBB ke COCO JSON dan YOLO-Seg


In [ ]:
def resolve_dataset_paths(input_dir, data_subdir, data_yaml_path):
    yaml_candidates = [
        Path(data_yaml_path),
        Path(input_dir) / "data.yaml",
        Path(data_subdir) / "data.yaml",
        Path("./panoptic-indoor-segmentation.v1i.yolov8-obb/data.yaml"),
    ]
    for search_root in [Path(input_dir), Path("/kaggle/input")]:
        if search_root.exists():
            yaml_candidates.extend(sorted(search_root.glob("**/data.yaml")))

    resolved_yaml = None
    for ycand in yaml_candidates:
        if ycand.is_file():
            resolved_yaml = ycand
            break
    if resolved_yaml is None:
        raise FileNotFoundError(f"data.yaml tidak ditemukan di {data_yaml_path} maupun {input_dir}.")

    dir_candidates = [
        Path(data_subdir),
        Path(input_dir),
        resolved_yaml.parent,
        Path(data_subdir) / "panoptic-indoor-segmentation.v1i.yolov8-obb",
        Path("./panoptic-indoor-segmentation.v1i.yolov8-obb"),
    ]
    for base in [Path(data_subdir), Path(input_dir)]:
        if base.exists():
            for child in sorted(base.iterdir()):
                if child.is_dir():
                    dir_candidates.append(child)
            for train_img_dir in sorted(base.glob("**/train/images")):
                dir_candidates.append(train_img_dir.parent.parent)

    resolved_root = None
    for dcand in dir_candidates:
        if (dcand / "train" / "images").exists():
            resolved_root = dcand
            break
    if resolved_root is None:
        raise FileNotFoundError(f"Folder split train/images tidak ditemukan di {data_subdir} maupun {input_dir}.")

    return resolved_root, resolved_yaml

DATASET_DIR, RESOLVED_YAML_PATH = resolve_dataset_paths(INPUT_DIR, DATA_SUBDIR, DATA_YAML_PATH)
with open(RESOLVED_YAML_PATH, "r", encoding="utf-8") as f:
    DATA_YAML = yaml.safe_load(f)

RAW_NAMES = DATA_YAML["names"]
if isinstance(RAW_NAMES, dict):
    CLASS_NAMES = [RAW_NAMES[k] for k in sorted(RAW_NAMES.keys(), key=int)]
else:
    CLASS_NAMES = list(RAW_NAMES)

NUM_CLASSES = len(CLASS_NAMES)
ID2LABEL = {idx: name for idx, name in enumerate(CLASS_NAMES)}
LABEL2ID = {name: idx for idx, name in ID2LABEL.items()}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def convert_yolo_obb_to_coco_and_yoloseg(dataset_root, coco_out_dir, yolo_out_dir):
    coco_paths = {}
    split_candidates = [
        ("train", ["train"]),
        ("valid", ["valid", "val"]),
        ("test", ["test"]),
    ]
    categories = [{"id": idx, "name": name, "supercategory": "indoor"} for idx, name in enumerate(CLASS_NAMES)]

    for split_key, folder_names in split_candidates:
        split_folder = None
        for fname in folder_names:
            if (dataset_root / fname / "images").exists():
                split_folder = fname
                break
        if split_folder is None:
            continue

        img_dir = dataset_root / split_folder / "images"
        lbl_dir = dataset_root / split_folder / "labels"

        yolo_img_dir = Path(yolo_out_dir) / split_key / "images"
        yolo_lbl_dir = Path(yolo_out_dir) / split_key / "labels"
        yolo_img_dir.mkdir(parents=True, exist_ok=True)
        yolo_lbl_dir.mkdir(parents=True, exist_ok=True)

        images_info = []
        annotations_info = []
        ann_id = 1
        image_files = sorted([p for p in img_dir.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

        for img_id, img_path in enumerate(tqdm(image_files, desc=f"Konversi {split_key}"), start=1):
            with Image.open(img_path) as im:
                width, height = im.size

            images_info.append({
                "id": img_id,
                "file_name": img_path.name,
                "path": str(img_path.resolve()),
                "width": width,
                "height": height,
            })

            target_img_link = yolo_img_dir / img_path.name
            if not target_img_link.exists():
                try:
                    os.symlink(img_path.resolve(), target_img_link)
                except OSError:
                    shutil.copy2(img_path, target_img_link)

            lbl_path = lbl_dir / f"{img_path.stem}.txt"
            yolo_seg_lines = []

            if lbl_path.exists():
                with open(lbl_path, "r", encoding="utf-8") as lf:
                    lines = [ln.strip() for ln in lf if ln.strip()]

                for line in lines:
                    parts = line.split()
                    if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
                        continue
                    cls_id = int(float(parts[0]))
                    if cls_id < 0 or cls_id >= NUM_CLASSES:
                        continue

                    norm_coords = np.array([float(x) for x in parts[1:]], dtype=np.float32).reshape(-1, 2)
                    norm_coords[:, 0] = np.clip(norm_coords[:, 0], 0.0, 1.0)
                    norm_coords[:, 1] = np.clip(norm_coords[:, 1], 0.0, 1.0)

                    pix_coords = norm_coords.copy()
                    pix_coords[:, 0] *= width
                    pix_coords[:, 1] *= height

                    x_min = float(np.min(pix_coords[:, 0]))
                    y_min = float(np.min(pix_coords[:, 1]))
                    x_max = float(np.max(pix_coords[:, 0]))
                    y_max = float(np.max(pix_coords[:, 1]))
                    box_w = x_max - x_min
                    box_h = y_max - y_min
                    if box_w <= 1.0 or box_h <= 1.0:
                        continue

                    area = float(cv2.contourArea(pix_coords.astype(np.float32)))
                    if area <= 1.0:
                        area = box_w * box_h

                    flat_poly = [round(float(v), 2) for v in pix_coords.flatten().tolist()]
                    annotations_info.append({
                        "id": ann_id,
                        "image_id": img_id,
                        "category_id": cls_id,
                        "segmentation": [flat_poly],
                        "area": round(area, 2),
                        "bbox": [round(x_min, 2), round(y_min, 2), round(box_w, 2), round(box_h, 2)],
                        "iscrowd": 0,
                    })
                    ann_id += 1

                    norm_str = " ".join(f"{v:.6f}" for v in norm_coords.flatten().tolist())
                    yolo_seg_lines.append(f"{cls_id} {norm_str}")

            with open(yolo_lbl_dir / f"{img_path.stem}.txt", "w", encoding="utf-8") as out_lf:
                out_lf.write("\n".join(yolo_seg_lines))

        coco_payload = {
            "info": {"description": "Panoptic Indoor Segmentation COCO Format"},
            "licenses": [],
            "categories": categories,
            "images": images_info,
            "annotations": annotations_info,
        }
        coco_json_path = Path(coco_out_dir) / f"instances_{split_key}.json"
        with open(coco_json_path, "w", encoding="utf-8") as cf:
            json.dump(coco_payload, cf)
        coco_paths[split_key] = str(coco_json_path)

    yolo_yaml_path = Path(yolo_out_dir) / "data_seg.yaml"
    yolo_yaml_payload = {
        "path": str(Path(yolo_out_dir).resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": NUM_CLASSES,
        "names": ID2LABEL,
    }
    with open(yolo_yaml_path, "w", encoding="utf-8") as yf:
        yaml.safe_dump(yolo_yaml_payload, yf, sort_keys=False)

    return coco_paths, str(yolo_yaml_path)

COCO_PATHS, YOLO_SEG_YAML = convert_yolo_obb_to_coco_and_yoloseg(DATASET_DIR, COCO_DIR, YOLO_SEG_DIR)
print("Dataset resolved  :", DATASET_DIR)
print("YAML resolved     :", RESOLVED_YAML_PATH)
print("Total classes     :", NUM_CLASSES)
print("COCO JSON paths   :", COCO_PATHS)
print("YOLO-Seg YAML     :", YOLO_SEG_YAML)


## 3. Pipeline Augmentasi (Skenario A, B, C) & Dataset PyTorch


In [ ]:
def get_augmentation_pipeline(scenario, img_size=IMG_SIZE):
    if scenario == "A":
        return A.Compose([A.Resize(img_size, img_size)])
    if scenario == "B":
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomScale(scale_limit=0.2, p=0.5),
            A.Resize(img_size, img_size),
        ])
    if scenario == "C":
        return A.Compose([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.0625,
                scale_limit=0.1,
                rotate_limit=15,
                border_mode=cv2.BORDER_CONSTANT,
                p=0.5,
            ),
            A.GaussianBlur(blur_limit=(3, 5), p=0.3),
            A.Resize(img_size, img_size),
        ])
    raise ValueError(f"Skenario tidak dikenal: {scenario}")


def get_yolo_custom_albumentations():
    return [
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.0625,
            scale_limit=0.1,
            rotate_limit=15,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.5,
        ),
        A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    ]


class CocoInstanceDataset(Dataset):
    def __init__(self, coco_json_path, mode="maskrcnn", scenario="A", img_size=IMG_SIZE, processor=None, is_train=True):
        self.coco = COCO(coco_json_path)
        self.mode = mode
        self.scenario = scenario
        self.img_size = img_size
        self.processor = processor
        self.is_train = is_train
        self.transform = get_augmentation_pipeline(scenario, img_size) if is_train else get_augmentation_pipeline("A", img_size)
        self.fallback_transform = get_augmentation_pipeline("A", img_size)

        all_img_ids = sorted(self.coco.getImgIds())
        if is_train:
            self.img_ids = [iid for iid in all_img_ids if len(self.coco.getAnnIds(imgIds=iid)) > 0]
        else:
            self.img_ids = all_img_ids

    def __len__(self):
        return len(self.img_ids)

    def _build_masks_and_labels(self, img_info, anns):
        h, w = img_info["height"], img_info["width"]
        masks = []
        cats = []
        for ann in anns:
            seg = ann.get("segmentation", [])
            if not seg:
                continue
            m = np.zeros((h, w), dtype=np.uint8)
            for poly in seg:
                pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
                cv2.fillPoly(m, [np.round(pts).astype(np.int32)], 1)
            if m.sum() > 1:
                masks.append(m)
                cats.append(int(ann["category_id"]))
        return masks, cats

    def _apply_transform(self, image_np, masks, cats, transform):
        if len(masks) > 0:
            augmented = transform(image=image_np, masks=masks)
            aug_img = augmented["image"]
            aug_masks = augmented["masks"]
        else:
            augmented = transform(image=image_np)
            aug_img = augmented["image"]
            aug_masks = []

        valid_masks = []
        valid_boxes = []
        valid_cats = []
        for m, c in zip(aug_masks, cats):
            ys, xs = np.where(m > 0)
            if len(xs) == 0 or len(ys) == 0:
                continue
            x1, y1, x2, y2 = float(xs.min()), float(ys.min()), float(xs.max()) + 1.0, float(ys.max()) + 1.0
            if (x2 - x1) <= 1.0 or (y2 - y1) <= 1.0:
                continue
            valid_masks.append((m > 0).astype(np.uint8))
            valid_boxes.append([x1, y1, x2, y2])
            valid_cats.append(c)

        return aug_img, valid_masks, valid_boxes, valid_cats

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        img_info = self.coco.loadImgs([img_id])[0]
        img_path = img_info["path"]
        image_np = np.array(Image.open(img_path).convert("RGB"), dtype=np.uint8)

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        raw_masks, raw_cats = self._build_masks_and_labels(img_info, anns)

        aug_img, valid_masks, valid_boxes, valid_cats = self._apply_transform(
            image_np, raw_masks, raw_cats, self.transform
        )
        if self.is_train and len(valid_masks) == 0 and len(raw_masks) > 0:
            aug_img, valid_masks, valid_boxes, valid_cats = self._apply_transform(
                image_np, raw_masks, raw_cats, self.fallback_transform
            )

        if self.mode == "maskrcnn":
            img_tensor = torch.from_numpy(aug_img).permute(2, 0, 1).float() / 255.0
            if len(valid_masks) > 0:
                boxes_tensor = torch.tensor(valid_boxes, dtype=torch.float32)
                labels_tensor = torch.tensor([c + 1 for c in valid_cats], dtype=torch.int64)
                masks_tensor = torch.tensor(np.stack(valid_masks, axis=0), dtype=torch.uint8)
                area_tensor = (boxes_tensor[:, 3] - boxes_tensor[:, 1]) * (boxes_tensor[:, 2] - boxes_tensor[:, 0])
            else:
                boxes_tensor = torch.zeros((0, 4), dtype=torch.float32)
                labels_tensor = torch.zeros((0,), dtype=torch.int64)
                masks_tensor = torch.zeros((0, aug_img.shape[0], aug_img.shape[1]), dtype=torch.uint8)
                area_tensor = torch.zeros((0,), dtype=torch.float32)

            target = {
                "boxes": boxes_tensor,
                "labels": labels_tensor,
                "masks": masks_tensor,
                "image_id": torch.tensor([img_id], dtype=torch.int64),
                "area": area_tensor,
                "iscrowd": torch.zeros((len(valid_cats),), dtype=torch.int64),
            }
            return img_tensor, target

        encoded = self.processor(
            images=aug_img,
            return_tensors="pt",
        )
        item = {}
        for k, v in encoded.items():
            if isinstance(v, torch.Tensor):
                item[k] = v.squeeze(0)
            else:
                item[k] = v[0]

        ph, pw = item["pixel_values"].shape[-2:]
        if "pixel_mask" not in item:
            item["pixel_mask"] = torch.ones((ph, pw), dtype=torch.int64)

        if len(valid_masks) > 0:
            mask_tensor = torch.from_numpy(np.stack(valid_masks, axis=0)).float()
            if mask_tensor.shape[-2:] != (ph, pw):
                mask_tensor = F.interpolate(mask_tensor.unsqueeze(0), size=(ph, pw), mode="nearest").squeeze(0)
            item["mask_labels"] = mask_tensor
            item["class_labels"] = torch.tensor(valid_cats, dtype=torch.int64)
        else:
            item["mask_labels"] = torch.zeros((0, ph, pw), dtype=torch.float32)
            item["class_labels"] = torch.zeros((0,), dtype=torch.int64)

        item["image_id"] = img_id
        item["orig_size"] = (img_info["height"], img_info["width"])
        return item


def maskrcnn_collate_fn(batch):
    return tuple(zip(*batch))


def mask2former_collate_fn(batch):
    return {
        "pixel_values": torch.stack([b["pixel_values"] for b in batch]),
        "pixel_mask": torch.stack([b["pixel_mask"] for b in batch]),
        "mask_labels": [b["mask_labels"] for b in batch],
        "class_labels": [b["class_labels"] for b in batch],
        "image_id": [b["image_id"] for b in batch],
        "orig_size": [b["orig_size"] for b in batch],
    }


## 4. Utilitas Benchmark GPU, Logger CSV/TensorBoard & Evaluator COCO mAP


In [ ]:
BENCHMARK_RECORDS = []


def query_nvidia_smi_vram():
    usage = {}
    try:
        res = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            check=True,
        )
        for line in res.stdout.strip().splitlines():
            parts = [p.strip() for p in line.split(",")]
            if len(parts) == 2:
                usage[int(parts[0])] = float(parts[1])
    except Exception:
        pass
    return usage


class GPUBenchmarkTracker:
    def __init__(self, model_name, scenario, gpu_ids):
        self.model_name = model_name
        self.scenario = scenario
        self.gpu_ids = gpu_ids
        self.start_time = None
        self.duration_sec = 0.0
        self.peak_vram_per_gpu_mb = {}
        self.total_peak_vram_mb = 0.0
        self._smi_peaks = {gid: 0.0 for gid in gpu_ids}
        self._stop_event = threading.Event()
        self._monitor_thread = None

    def _poll_smi(self):
        while not self._stop_event.is_set():
            snap = query_nvidia_smi_vram()
            for gid in self.gpu_ids:
                if gid in snap:
                    self._smi_peaks[gid] = max(self._smi_peaks[gid], snap[gid])
            self._stop_event.wait(0.5)

    def __enter__(self):
        for gid in self.gpu_ids:
            try:
                torch.cuda.reset_peak_memory_stats(gid)
                torch.cuda.synchronize(gid)
            except Exception:
                pass
        if self.gpu_ids:
            self._monitor_thread = threading.Thread(target=self._poll_smi, daemon=True)
            self._monitor_thread.start()
        self.start_time = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self._stop_event.set()
        if self._monitor_thread is not None:
            self._monitor_thread.join(timeout=2.0)
        for gid in self.gpu_ids:
            torch_peak_mb = 0.0
            try:
                torch.cuda.synchronize(gid)
                torch_peak_mb = torch.cuda.max_memory_allocated(gid) / (1024 ** 2)
            except Exception:
                pass
            smi_peak_mb = self._smi_peaks.get(gid, 0.0)
            self.peak_vram_per_gpu_mb[f"gpu_{gid}_mb"] = round(max(torch_peak_mb, smi_peak_mb), 2)
        self.duration_sec = round(time.perf_counter() - self.start_time, 2)
        self.total_peak_vram_mb = round(sum(self.peak_vram_per_gpu_mb.values()), 2)

    def to_record(self, metrics):
        return {
            "model": self.model_name,
            "scenario": self.scenario,
            "num_gpus": len(self.gpu_ids),
            "wall_clock_time_sec": self.duration_sec,
            "total_peak_vram_mb": self.total_peak_vram_mb,
            "peak_vram_per_gpu_mb": self.peak_vram_per_gpu_mb,
            "mAP50": round(float(metrics.get("mAP50", 0.0)), 4),
            "mAP50_95": round(float(metrics.get("mAP50-95", 0.0)), 4),
            "best_val_loss": round(float(metrics.get("best_val_loss", 0.0)), 4),
        }


def build_dual_gpu_replicas(primary_wrapper, gpu_ids):
    replicas = [primary_wrapper]
    if len(gpu_ids) > 1:
        for gid in gpu_ids[1:]:
            rep = copy.deepcopy(primary_wrapper).to(torch.device(f"cuda:{gid}"))
            replicas.append(rep)
    return replicas


def merge_replica_grads_to_primary(replicas, primary_device):
    if len(replicas) <= 1:
        return
    primary = replicas[0]
    for rep in replicas[1:]:
        for p0, p1 in zip(primary.parameters(), rep.parameters()):
            if p1.grad is not None:
                grad_moved = p1.grad.to(primary_device, non_blocking=True)
                if p0.grad is None:
                    p0.grad = grad_moved
                else:
                    p0.grad.add_(grad_moved)
                p1.grad = None


def sync_replicas_from_primary(replicas):
    if len(replicas) <= 1:
        return
    primary = replicas[0]
    with torch.no_grad():
        for rep in replicas[1:]:
            for p0, p1 in zip(primary.parameters(), rep.parameters()):
                p1.copy_(p0.to(p1.device, non_blocking=True))
            for b0, b1 in zip(primary.buffers(), rep.buffers()):
                if b0.dtype == b1.dtype:
                    b1.copy_(b0.to(b1.device, non_blocking=True))


class MetricLogger:
    def __init__(self, run_name, log_root=LOG_DIR):
        self.run_dir = os.path.join(log_root, run_name)
        os.makedirs(self.run_dir, exist_ok=True)
        self.csv_path = os.path.join(self.run_dir, "results.csv")
        self.writer = SummaryWriter(log_dir=os.path.join(self.run_dir, "tensorboard"))
        self.rows = []

    def log_epoch(self, epoch, metrics_dict):
        row = {"epoch": epoch, **metrics_dict}
        self.rows.append(row)
        pd.DataFrame(self.rows).to_csv(self.csv_path, index=False)
        for key, val in metrics_dict.items():
            if isinstance(val, (int, float)):
                self.writer.add_scalar(key, float(val), epoch)
        self.writer.flush()

    def close(self):
        self.writer.close()


def evaluate_coco_instance_map(predictions, coco_gt_path):
    if len(predictions) == 0:
        return {"mAP50": 0.0, "mAP50-95": 0.0}
    coco_gt = COCO(coco_gt_path)
    coco_dt = coco_gt.loadRes(predictions)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="segm")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    return {
        "mAP50-95": float(coco_eval.stats[0]),
        "mAP50": float(coco_eval.stats[1]),
    }


def encode_binary_mask_rle(binary_mask):
    rle = mask_util.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle


def save_benchmark_files(records):
    with open(BENCHMARK_JSON_PATH, "w", encoding="utf-8") as jf:
        json.dump(records, jf, indent=2)
    flat_rows = []
    for rec in records:
        row = {k: v for k, v in rec.items() if k != "peak_vram_per_gpu_mb"}
        for gk, gv in rec.get("peak_vram_per_gpu_mb", {}).items():
            row[gk] = gv
        flat_rows.append(row)
    pd.DataFrame(flat_rows).to_csv(BENCHMARK_CSV_PATH, index=False)


## 5. Model 1 — Mask2Former Large (`facebook/mask2former-swin-large-coco-instance`)


In [ ]:
MASK2FORMER_MODEL_ID = "facebook/mask2former-swin-large-coco-instance"


class FP32Mask2FormerPixelDecoderEncoder(Mask2FormerPixelDecoderEncoderOnly):
    def forward(
        self,
        inputs_embeds=None,
        attention_mask=None,
        position_embeddings=None,
        spatial_shapes_list=None,
        level_start_index=None,
        valid_ratios=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        if inputs_embeds is not None:
            inputs_embeds = inputs_embeds.float()
        if position_embeddings is not None:
            position_embeddings = position_embeddings.float()
        if valid_ratios is not None:
            valid_ratios = valid_ratios.float()
        with torch.cuda.amp.autocast(enabled=False):
            return super().forward(
                inputs_embeds=inputs_embeds,
                attention_mask=attention_mask,
                position_embeddings=position_embeddings,
                spatial_shapes_list=spatial_shapes_list,
                level_start_index=level_start_index,
                valid_ratios=valid_ratios,
                output_attentions=output_attentions,
                output_hidden_states=output_hidden_states,
                return_dict=return_dict,
            )


def apply_fp32_encoder_patch(model):
    if hasattr(model, "model") and hasattr(model.model, "pixel_level_module") and hasattr(model.model.pixel_level_module, "decoder"):
        enc = model.model.pixel_level_module.decoder.encoder
        if isinstance(enc, Mask2FormerPixelDecoderEncoderOnly):
            enc.__class__ = FP32Mask2FormerPixelDecoderEncoder


class Mask2FormerAMPWrapper(nn.Module):
    def __init__(self, model, use_amp=True):
        super().__init__()
        self.model = model
        self.use_amp = use_amp

    def forward_with_outputs(self, pixel_values, pixel_mask, mask_labels=None, class_labels=None):
        dev = pixel_values.device
        dev_ctx = torch.cuda.device(dev) if dev.type == "cuda" else contextlib.nullcontext()
        with dev_ctx:
            with torch.cuda.amp.autocast(enabled=self.use_amp and dev.type == "cuda"):
                outputs = self.model.model(
                    pixel_values=pixel_values,
                    pixel_mask=pixel_mask,
                    output_hidden_states=self.model.config.use_auxiliary_loss,
                    return_dict=True,
                )
                class_queries_logits = tuple(
                    self.model.class_predictor(d.transpose(0, 1)).float()
                    for d in outputs.transformer_decoder_intermediate_states
                )
                masks_queries_logits = tuple(m.float() for m in outputs.masks_queries_logits)

            loss = None
            if mask_labels is not None and class_labels is not None and all(m.shape[0] > 0 for m in mask_labels):
                auxiliary_logits = self.model.get_auxiliary_logits(class_queries_logits, masks_queries_logits)
                ml_fp32 = [m.float() for m in mask_labels]
                loss_dict = self.model.get_loss_dict(
                    masks_queries_logits=masks_queries_logits[-1],
                    class_queries_logits=class_queries_logits[-1],
                    mask_labels=ml_fp32,
                    class_labels=class_labels,
                    auxiliary_predictions=auxiliary_logits,
                )
                loss = self.model.get_loss(loss_dict)
        return loss, class_queries_logits[-1], masks_queries_logits[-1]

    def forward(self, pixel_values, pixel_mask, mask_labels, class_labels):
        loss, _, _ = self.forward_with_outputs(pixel_values, pixel_mask, mask_labels, class_labels)
        return loss


def split_mask2former_batch_for_gpus(batch, gpu_ids):
    bsz = batch["pixel_values"].size(0)
    if len(gpu_ids) <= 1 or bsz < 2:
        dev = PRIMARY_DEVICE
        pv = batch["pixel_values"].to(dev, non_blocking=True)
        pm = batch["pixel_mask"].to(dev, non_blocking=True)
        ml = [m.to(dev, non_blocking=True) for m in batch["mask_labels"]]
        cl = [c.to(dev, non_blocking=True) for c in batch["class_labels"]]
        return [(pv, pm, ml, cl)]

    n_chunks = min(len(gpu_ids), bsz)
    indices = np.array_split(np.arange(bsz), n_chunks)
    chunk_inputs = []
    for gid, idxs in zip(gpu_ids[:n_chunks], indices):
        dev = torch.device(f"cuda:{gid}")
        idx_list = idxs.tolist()
        pv = batch["pixel_values"][idx_list].to(dev, non_blocking=True)
        pm = batch["pixel_mask"][idx_list].to(dev, non_blocking=True)
        ml = [batch["mask_labels"][i].to(dev, non_blocking=True) for i in idx_list]
        cl = [batch["class_labels"][i].to(dev, non_blocking=True) for i in idx_list]
        chunk_inputs.append((pv, pm, ml, cl))
    return chunk_inputs


@torch.no_grad()
def evaluate_mask2former(wrapper, processor, val_loader, coco_gt_path):
    wrapper.eval()
    predictions = []
    total_val_loss = 0.0
    n_batches = 0

    for batch in tqdm(val_loader, desc="Validasi Mask2Former", leave=False):
        pv = batch["pixel_values"].to(PRIMARY_DEVICE)
        pm = batch["pixel_mask"].to(PRIMARY_DEVICE)
        ml = [m.to(PRIMARY_DEVICE) for m in batch["mask_labels"]]
        cl = [c.to(PRIMARY_DEVICE) for c in batch["class_labels"]]

        loss, cls_logits, mask_logits = wrapper.forward_with_outputs(pv, pm, ml, cl)
        if loss is not None:
            total_val_loss += float(loss.item())
            n_batches += 1

        probs = cls_logits.softmax(-1)[..., :-1]
        num_q, num_c = probs.shape[1], probs.shape[2]

        for b_idx, (img_id, orig_hw) in enumerate(zip(batch["image_id"], batch["orig_size"])):
            orig_h, orig_w = int(orig_hw[0]), int(orig_hw[1])
            flat_scores, topk_flat_idx = probs[b_idx].reshape(-1).topk(min(100, num_q * num_c))
            q_idx = torch.div(topk_flat_idx, num_c, rounding_mode="floor")
            c_idx = topk_flat_idx % num_c
            keep_cls = flat_scores >= 0.05
            if not torch.any(keep_cls):
                continue

            q_idx = q_idx[keep_cls]
            c_idx = c_idx[keep_cls]
            cls_scores = flat_scores[keep_cls]
            masks_k = mask_logits[b_idx, q_idx]
            masks_resized = F.interpolate(
                masks_k.unsqueeze(0),
                size=(orig_h, orig_w),
                mode="bilinear",
                align_corners=False,
            ).squeeze(0)
            mask_probs = masks_resized.sigmoid()
            bin_masks_t = (mask_probs > 0.5).float()
            maskness = (mask_probs * bin_masks_t).flatten(1).sum(1) / (bin_masks_t.flatten(1).sum(1) + 1e-6)
            final_scores = (cls_scores * maskness).detach().cpu().numpy()
            bin_masks = bin_masks_t.detach().cpu().numpy().astype(np.uint8)
            labels_np = c_idx.detach().cpu().numpy()

            for sc, lbl, bm in zip(final_scores, labels_np, bin_masks):
                if sc < 0.05 or bm.sum() == 0:
                    continue
                predictions.append({
                    "image_id": int(img_id),
                    "category_id": int(lbl),
                    "segmentation": encode_binary_mask_rle(bm),
                    "score": float(sc),
                })

    coco_metrics = evaluate_coco_instance_map(predictions, coco_gt_path)
    coco_metrics["val_loss"] = total_val_loss / max(1, n_batches)
    return coco_metrics


def train_mask2former_scenario(scenario):
    run_name = f"mask2former_large_scenario_{scenario}"
    logger = MetricLogger(run_name)
    ckpt_dir = os.path.join(CHECKPOINT_DIR, run_name)
    os.makedirs(ckpt_dir, exist_ok=True)

    processor = Mask2FormerImageProcessor.from_pretrained(
        MASK2FORMER_MODEL_ID,
        do_reduce_labels=False,
        ignore_index=255,
        size={"height": IMG_SIZE, "width": IMG_SIZE},
    )
    model = Mask2FormerForUniversalSegmentation.from_pretrained(
        MASK2FORMER_MODEL_ID,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )
    apply_fp32_encoder_patch(model)
    model = model.to(PRIMARY_DEVICE)

    wrapper = Mask2FormerAMPWrapper(model, use_amp=USE_AMP)
    replicas = build_dual_gpu_replicas(wrapper, GPU_IDS)

    train_ds = CocoInstanceDataset(
        COCO_PATHS["train"], mode="mask2former", scenario=scenario, img_size=IMG_SIZE, processor=processor, is_train=True
    )
    val_ds = CocoInstanceDataset(
        COCO_PATHS["valid"], mode="mask2former", scenario="A", img_size=IMG_SIZE, processor=processor, is_train=False
    )

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        collate_fn=mask2former_collate_fn, pin_memory=False, drop_last=False
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        collate_fn=mask2former_collate_fn, pin_memory=False
    )

    backbone_params = [p for n, p in model.named_parameters() if 'pixel_level_module.encoder' in n and p.requires_grad]
    head_params = [p for n, p in model.named_parameters() if 'pixel_level_module.encoder' not in n and p.requires_grad]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': LR_MASK2FORMER * 0.1},
        {'params': head_params, 'lr': LR_MASK2FORMER},
    ], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_map50_95 = -1.0
    best_val_loss = float("inf")
    best_metrics = {"mAP50": 0.0, "mAP50-95": 0.0, "best_val_loss": 0.0}

    with GPUBenchmarkTracker("Mask2Former-Large", scenario, GPU_IDS) as tracker:
        for epoch in range(1, EPOCHS + 1):
            for rep in replicas:
                rep.train()
                rep.zero_grad(set_to_none=True)
            train_loss_sum = 0.0
            optimizer.zero_grad(set_to_none=True)

            pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Mask2Former [{scenario}] Ep {epoch}/{EPOCHS}")
            for step, batch in pbar:
                chunk_inputs = split_mask2former_batch_for_gpus(batch, GPU_IDS)
                n_chunks = len(chunk_inputs)
                step_loss_val = 0.0
                for c_idx, chunk in enumerate(chunk_inputs):
                    loss_i = replicas[c_idx](*chunk)
                    step_loss_val += float(loss_i.item()) / n_chunks
                    scaler.scale(loss_i / (n_chunks * GRAD_ACCUM_STEPS)).backward()

                if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                    merge_replica_grads_to_primary(replicas, PRIMARY_DEVICE)
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    sync_replicas_from_primary(replicas)

                train_loss_sum += step_loss_val
                pbar.set_postfix({"loss": f"{train_loss_sum / (step + 1):.4f}"})

            scheduler.step()
            avg_train_loss = train_loss_sum / max(1, len(train_loader))
            val_metrics = evaluate_mask2former(wrapper, processor, val_loader, COCO_PATHS["valid"])
            epoch_metrics = {
                "train_loss": round(avg_train_loss, 4),
                "val_loss": round(val_metrics["val_loss"], 4),
                "mAP50": round(val_metrics["mAP50"], 4),
                "mAP50-95": round(val_metrics["mAP50-95"], 4),
            }
            logger.log_epoch(epoch, epoch_metrics)

            if val_metrics["val_loss"] < best_val_loss:
                best_val_loss = val_metrics["val_loss"]

            if val_metrics["mAP50-95"] >= best_map50_95:
                best_map50_95 = val_metrics["mAP50-95"]
                best_metrics = {
                    "mAP50": val_metrics["mAP50"],
                    "mAP50-95": val_metrics["mAP50-95"],
                    "best_val_loss": best_val_loss,
                }
                model.save_pretrained(ckpt_dir)
                processor.save_pretrained(ckpt_dir)

    logger.close()
    record = tracker.to_record(best_metrics)
    BENCHMARK_RECORDS.append(record)
    save_benchmark_files(BENCHMARK_RECORDS)

    del replicas, model, wrapper, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record


MASK2FORMER_RESULTS = {}
for sc in ["A", "B", "C"]:
    MASK2FORMER_RESULTS[sc] = train_mask2former_scenario(sc)
    print(f"Selesai Mask2Former Skenario {sc}:", MASK2FORMER_RESULTS[sc])


## 6. Model 2 — Mask R-CNN Large (`ResNet-101-FPN`)


In [ ]:
class MaskRCNNAMPWrapper(nn.Module):
    def __init__(self, model, use_amp=True):
        super().__init__()
        self.model = model
        self.use_amp = use_amp

    def forward(self, images, targets):
        dev = images[0].device
        dev_ctx = torch.cuda.device(dev) if dev.type == "cuda" else contextlib.nullcontext()
        with dev_ctx:
            with torch.cuda.amp.autocast(enabled=self.use_amp and dev.type == "cuda"):
                loss_dict = self.model(images, targets)
                return sum(loss for loss in loss_dict.values())


def build_maskrcnn_large(num_classes):
    backbone = resnet_fpn_backbone(
        backbone_name="resnet101",
        weights=ResNet101_Weights.DEFAULT,
        trainable_layers=5,
    )
    model = MaskRCNN(
        backbone,
        num_classes=num_classes,
        min_size=IMG_SIZE,
        max_size=IMG_SIZE,
    )
    ref_model = maskrcnn_resnet50_fpn_v2(weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
    model.rpn.load_state_dict(ref_model.rpn.state_dict(), strict=False)
    model.roi_heads.box_head.load_state_dict(ref_model.roi_heads.box_head.state_dict(), strict=False)
    model.roi_heads.mask_head.load_state_dict(ref_model.roi_heads.mask_head.state_dict(), strict=False)
    del ref_model
    in_features_box = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features_box, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model


def split_maskrcnn_batch_for_gpus(images, targets, gpu_ids):
    bsz = len(images)
    if len(gpu_ids) <= 1 or bsz < 2:
        dev = PRIMARY_DEVICE
        img_dev = [img.to(dev, non_blocking=True) for img in images]
        tgt_dev = [{k: v.to(dev, non_blocking=True) for k, v in t.items()} for t in targets]
        return [(img_dev, tgt_dev)]

    n_chunks = min(len(gpu_ids), bsz)
    indices = np.array_split(np.arange(bsz), n_chunks)
    chunk_inputs = []
    for gid, idxs in zip(gpu_ids[:n_chunks], indices):
        dev = torch.device(f"cuda:{gid}")
        idx_list = idxs.tolist()
        img_chunk = [images[i].to(dev, non_blocking=True) for i in idx_list]
        tgt_chunk = [{k: v.to(dev, non_blocking=True) for k, v in targets[i].items()} for i in idx_list]
        chunk_inputs.append((img_chunk, tgt_chunk))
    return chunk_inputs


@torch.no_grad()
def evaluate_maskrcnn(model, val_loader, coco_gt_path, val_coco_obj):
    model.eval()
    predictions = []

    for images, targets in tqdm(val_loader, desc="Validasi Mask R-CNN", leave=False):
        img_list = [img.to(PRIMARY_DEVICE) for img in images]
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(img_list)

        for tgt, out in zip(targets, outputs):
            img_id = int(tgt["image_id"].item())
            img_info = val_coco_obj.loadImgs([img_id])[0]
            orig_h, orig_w = img_info["height"], img_info["width"]
            scores = out["scores"].detach().cpu().numpy()
            labels = out["labels"].detach().cpu().numpy()
            masks = out["masks"].detach().cpu().numpy()

            for score, label, mask in zip(scores, labels, masks):
                if float(score) < 0.05:
                    continue
                bin_mask = (mask[0] > 0.5).astype(np.uint8)
                if bin_mask.shape != (orig_h, orig_w):
                    bin_mask = cv2.resize(bin_mask, (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)
                if bin_mask.sum() == 0:
                    continue
                cat_id = int(label) - 1
                if cat_id < 0 or cat_id >= NUM_CLASSES:
                    continue
                predictions.append({
                    "image_id": img_id,
                    "category_id": cat_id,
                    "segmentation": encode_binary_mask_rle(bin_mask),
                    "score": float(score),
                })

    return evaluate_coco_instance_map(predictions, coco_gt_path)


def train_maskrcnn_scenario(scenario):
    run_name = f"maskrcnn_large_scenario_{scenario}"
    logger = MetricLogger(run_name)
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{run_name}_best.pth")

    model = build_maskrcnn_large(NUM_CLASSES + 1).to(PRIMARY_DEVICE)
    wrapper = MaskRCNNAMPWrapper(model, use_amp=USE_AMP)
    replicas = build_dual_gpu_replicas(wrapper, GPU_IDS)

    train_ds = CocoInstanceDataset(
        COCO_PATHS["train"], mode="maskrcnn", scenario=scenario, img_size=IMG_SIZE, is_train=True
    )
    val_ds = CocoInstanceDataset(
        COCO_PATHS["valid"], mode="maskrcnn", scenario="A", img_size=IMG_SIZE, is_train=False
    )

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        collate_fn=maskrcnn_collate_fn, pin_memory=False
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
        collate_fn=maskrcnn_collate_fn, pin_memory=False
    )

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR_MASKRCNN, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    best_map50_95 = -1.0
    best_metrics = {"mAP50": 0.0, "mAP50-95": 0.0, "best_val_loss": 0.0}

    with GPUBenchmarkTracker("Mask-RCNN-Large", scenario, GPU_IDS) as tracker:
        for epoch in range(1, EPOCHS + 1):
            for rep in replicas:
                rep.train()
                rep.zero_grad(set_to_none=True)
            train_loss_sum = 0.0
            optimizer.zero_grad(set_to_none=True)

            pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Mask R-CNN [{scenario}] Ep {epoch}/{EPOCHS}")
            for step, (images, targets) in pbar:
                chunk_inputs = split_maskrcnn_batch_for_gpus(images, targets, GPU_IDS)
                n_chunks = len(chunk_inputs)
                step_loss_val = 0.0
                for c_idx, chunk in enumerate(chunk_inputs):
                    loss_i = replicas[c_idx](*chunk)
                    step_loss_val += float(loss_i.item()) / n_chunks
                    scaler.scale(loss_i / (n_chunks * GRAD_ACCUM_STEPS)).backward()

                if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                    merge_replica_grads_to_primary(replicas, PRIMARY_DEVICE)
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(params, max_norm=5.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    sync_replicas_from_primary(replicas)

                train_loss_sum += step_loss_val
                pbar.set_postfix({"loss": f"{train_loss_sum / (step + 1):.4f}"})

            scheduler.step()
            avg_train_loss = train_loss_sum / max(1, len(train_loader))
            val_metrics = evaluate_maskrcnn(model, val_loader, COCO_PATHS["valid"], val_ds.coco)
            epoch_metrics = {
                "train_loss": round(avg_train_loss, 4),
                "mAP50": round(val_metrics["mAP50"], 4),
                "mAP50-95": round(val_metrics["mAP50-95"], 4),
            }
            logger.log_epoch(epoch, epoch_metrics)

            if val_metrics["mAP50-95"] >= best_map50_95:
                best_map50_95 = val_metrics["mAP50-95"]
                best_metrics = {
                    "mAP50": val_metrics["mAP50"],
                    "mAP50-95": val_metrics["mAP50-95"],
                    "best_val_loss": avg_train_loss,
                }
                torch.save(model.state_dict(), ckpt_path)

    logger.close()
    record = tracker.to_record(best_metrics)
    BENCHMARK_RECORDS.append(record)
    save_benchmark_files(BENCHMARK_RECORDS)

    del replicas, model, wrapper, optimizer, scheduler, scaler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record


MASKRCNN_RESULTS = {}
for sc in ["A", "B", "C"]:
    MASKRCNN_RESULTS[sc] = train_maskrcnn_scenario(sc)
    print(f"Selesai Mask R-CNN Skenario {sc}:", MASKRCNN_RESULTS[sc])


## 7. Model 3 — YOLOv26-seg Large (`yolo26l-seg`)


In [ ]:
def load_yolo26_seg_model():
    try:
        return YOLO("yolo26l-seg.pt"), "yolo26l-seg"
    except Exception:
        return YOLO("yolo11l-seg.pt"), "yolo11l-seg-fallback"


def train_yolo_scenario(scenario):
    run_name = f"yolo26l_seg_scenario_{scenario}"
    yolo_project_dir = os.path.join(LOG_DIR, "ultralytics")
    os.makedirs(yolo_project_dir, exist_ok=True)

    model, resolved_name = load_yolo26_seg_model()
    yolo_device = GPU_IDS if len(GPU_IDS) > 1 else (0 if torch.cuda.is_available() else "cpu")

    train_kwargs = {
        "data": YOLO_SEG_YAML,
        "epochs": EPOCHS,
        "imgsz": IMG_SIZE,
        "batch": BATCH_SIZE * max(1, len(GPU_IDS)),
        "device": yolo_device,
        "amp": USE_AMP,
        "workers": NUM_WORKERS,
        "project": yolo_project_dir,
        "name": run_name,
        "exist_ok": True,
        "plots": True,
        "seed": SEED,
        "overlap_mask": False,
        "cos_lr": True,
    }

    if scenario == "A":
        train_kwargs.update({
            "hsv_h": 0.0,
            "hsv_s": 0.0,
            "hsv_v": 0.0,
            "degrees": 0.0,
            "translate": 0.0,
            "scale": 0.0,
            "shear": 0.0,
            "perspective": 0.0,
            "flipud": 0.0,
            "fliplr": 0.0,
            "bgr": 0.0,
            "mosaic": 0.0,
            "mixup": 0.0,
            "cutmix": 0.0,
            "copy_paste": 0.0,
            "auto_augment": None,
            "erasing": 0.0,
            "augmentations": [],
        })
    elif scenario == "C":
        train_kwargs.update({
            "mosaic": 0.0,
            "mixup": 0.0,
            "augmentations": get_yolo_custom_albumentations(),
        })

    with GPUBenchmarkTracker(f"YOLOv26l-seg ({resolved_name})", scenario, GPU_IDS) as tracker:
        train_results = model.train(**train_kwargs)

    save_dir = Path(train_results.save_dir) if hasattr(train_results, "save_dir") else Path(yolo_project_dir) / run_name
    best_weights = save_dir / "weights" / "best.pt"
    if best_weights.exists():
        shutil.copy2(best_weights, os.path.join(CHECKPOINT_DIR, f"{run_name}_best.pt"))

    results_csv = save_dir / "results.csv"
    map50 = 0.0
    map50_95 = 0.0
    best_loss = 0.0
    if results_csv.exists():
        df = pd.read_csv(results_csv)
        df.columns = [c.strip() for c in df.columns]
        m50_cols = [c for c in df.columns if "mAP50(M)" in c]
        m5095_cols = [c for c in df.columns if "mAP50-95(M)" in c]
        loss_cols = [c for c in df.columns if "val/seg_loss" in c]
        if m50_cols:
            map50 = float(df[m50_cols[0]].max())
        if m5095_cols:
            map50_95 = float(df[m5095_cols[0]].max())
        if loss_cols:
            best_loss = float(df[loss_cols[0]].min())

    record = tracker.to_record({
        "mAP50": map50,
        "mAP50-95": map50_95,
        "best_val_loss": best_loss,
    })
    record["weights_path"] = str(os.path.join(CHECKPOINT_DIR, f"{run_name}_best.pt"))
    BENCHMARK_RECORDS.append(record)
    save_benchmark_files(BENCHMARK_RECORDS)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record


YOLO_RESULTS = {}
for sc in ["A", "B", "C"]:
    YOLO_RESULTS[sc] = train_yolo_scenario(sc)
    print(f"Selesai YOLOv26l-seg Skenario {sc}:", YOLO_RESULTS[sc])


## 8. Rekapitulasi GPU Benchmark (`gpu_benchmark.json` / `.csv`) & Pembuatan `experiment_logbook.md`


In [ ]:
save_benchmark_files(BENCHMARK_RECORDS)
df_bench = pd.read_csv(BENCHMARK_CSV_PATH)
print(df_bench.to_string(index=False))


def generate_experiment_logbook(records, output_path):
    lines = [
        "# Experiment Logbook: Instance Segmentation Benchmark (Kaggle 2x T4)",
        "",
        f"- **Dataset**: `{DATASET_DIR}` (`{NUM_CLASSES}` classes)",
        f"- **Hardware**: `{len(GPU_IDS)}x GPU` (`{GPU_IDS}`), `IMG_SIZE={IMG_SIZE}`, `EPOCHS={EPOCHS}`, `FP16 AMP={USE_AMP}`",
        "",
        "## 1. Tabel Perbandingan Metrik & Benchmark GPU",
        "",
        "| Model | Skenario Augmentasi | mAP50 | mAP50-95 | Wall-Clock Time (s) | Total Peak VRAM (MB) |",
        "| :--- | :---: | :---: | :---: | :---: | :---: |",
    ]
    for rec in records:
        lines.append(
            f"| {rec['model']} | {rec['scenario']} | {rec['mAP50']:.4f} | {rec['mAP50_95']:.4f} | "
            f"{rec['wall_clock_time_sec']:.2f} | {rec['total_peak_vram_mb']:.2f} |"
        )
    lines.extend([
        "",
        "## 2. Kendala Teknis (Technical Issues & Resolutions)",
        "",
        "| Tanggal / Fase | Model / Skenario | Kendala yang Ditemukan | Solusi / Workaround yang Diterapkan |",
        "| :--- | :--- | :--- | :--- |",
        "| Preprocessing | Semua Model | Format dataset awal `yolov8-obb` belum kompatibel dengan Mask2Former & Mask R-CNN | Konversi otomatis ke COCO JSON poligon/mask & YOLO-Seg YAML di `WORKING_DIR` |",
        "| Multi-GPU Training | Mask2Former / Mask R-CNN | [Catat kendala OOM / sinkronisasi batch jika ada] | [Catat solusi penyesuaian batch size / gradient accumulation] |",
        "| Export TFLite | Mask2Former / Mask R-CNN / YOLO | [Catat unsupported ops ONNX -> TFLite yang muncul] | [Catat fallback SELECT_TF_OPS / Flex delegate / FP16] |",
        "",
        "## 3. Keputusan Arsitektural & Analisis Augmentasi",
        "",
        "- **Perbandingan Skenario A vs B vs C**:",
        "  - *Observasi Skenario A (Raw)*: [Isi hasil konvergensi & potensi overfitting]",
        "  - *Observasi Skenario B (Default Framework)*: [Isi dampak augmentasi bawaan]",
        "  - *Observasi Skenario C (Custom Albumentations Indoor)*: [Isi efektivitas RandomBrightnessContrast, HorizontalFlip, ShiftScaleRotate, GaussianBlur]",
        "- **Pemilihan Model Terbaik untuk Deployment**:",
        "  - *Akurasi vs Latensi vs Ukuran `.tflite`*: [Isi rekomendasi model terbaik berdasarkan tabel di atas]",
        "",
        "## 4. Lokasi Artefak Output",
        "",
        f"- Benchmark JSON: `{BENCHMARK_JSON_PATH}`",
        f"- Benchmark CSV: `{BENCHMARK_CSV_PATH}`",
        f"- Logs & TensorBoard: `{LOG_DIR}`",
        f"- Model Checkpoints: `{CHECKPOINT_DIR}`",
        f"- TFLite Exports: `{TFLITE_EXPORT_DIR}`",
        "",
    ])
    with open(output_path, "w", encoding="utf-8") as lf:
        lf.write("\n".join(lines))

generate_experiment_logbook(BENCHMARK_RECORDS, LOGBOOK_PATH)
print("Experiment logbook disimpan ke:", LOGBOOK_PATH)


## 9. Ekspor Best Weights ke TensorFlow Lite (`.tflite` FP16 / INT8) & Penanganan Unsupported Ops


In [ ]:
class ExportableMask2FormerWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        outputs = self.model(pixel_values=pixel_values)
        return outputs.class_queries_logits, outputs.masks_queries_logits


class ExportableMaskRCNNWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, pixel_values):
        preds = self.model([pixel_values[0]])
        out = preds[0]
        return out["boxes"], out["labels"], out["scores"], out["masks"]


def export_to_onnx_compat(wrapper, dummy_input, onnx_path, input_names, output_names):
    export_kwargs = {
        "opset_version": 16,
        "do_constant_folding": True,
        "input_names": input_names,
        "output_names": output_names,
    }
    if "dynamo" in inspect.signature(torch.onnx.export).parameters:
        export_kwargs["dynamo"] = False
    torch.onnx.export(wrapper, dummy_input, onnx_path, **export_kwargs)


def select_best_scenario(records, model_prefix):
    filtered = [r for r in records if r["model"].startswith(model_prefix)]
    if not filtered:
        return "C"
    best = max(filtered, key=lambda x: x["mAP50_95"])
    return best["scenario"]


def infer_saved_model_input_shape(saved_model_dir):
    loaded = tf.saved_model.load(saved_model_dir)
    sig = loaded.signatures.get("serving_default")
    if sig is None and loaded.signatures:
        sig = next(iter(loaded.signatures.values()))
    if sig is not None and sig.inputs:
        raw_shape = sig.inputs[0].shape.as_list()
        return tuple(d if (d is not None and d > 0) else 1 for d in raw_shape)
    return (1, IMG_SIZE, IMG_SIZE, 3)


def convert_onnx_to_tflite_with_fallback(onnx_path, model_tag):
    saved_model_dir = os.path.join(TFLITE_EXPORT_DIR, f"{model_tag}_saved_model")
    if os.path.exists(saved_model_dir):
        shutil.rmtree(saved_model_dir)

    cmd = [
        "onnx2tf",
        "-i", onnx_path,
        "-o", saved_model_dir,
        "-osd",
        "-cotof",
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    export_status = {
        "model_tag": model_tag,
        "onnx_path": onnx_path,
        "onnx2tf_returncode": proc.returncode,
        "fp16_tflite": None,
        "int8_tflite": None,
        "ops_fallback_used": [],
        "error": None,
    }

    if proc.returncode != 0 or not os.path.exists(saved_model_dir):
        export_status["error"] = proc.stderr[-1000:] if proc.stderr else "onnx2tf failed"
        return export_status

    input_shape = infer_saved_model_input_shape(saved_model_dir)

    def run_tflite_conversion(quant_mode, allow_flex_ops):
        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        if quant_mode == "fp16":
            converter.target_spec.supported_types = [tf.float16]
        else:
            def representative_dataset():
                for _ in range(5):
                    yield [np.random.rand(*input_shape).astype(np.float32)]
            converter.representative_dataset = representative_dataset
            converter.target_spec.supported_types = []

        if allow_flex_ops:
            converter.target_spec.supported_ops = [
                tf.lite.OpsSet.TFLITE_BUILTINS,
                tf.lite.OpsSet.SELECT_TF_OPS,
            ]
        else:
            converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
        return converter.convert()

    for qmode in ["fp16", "int8"]:
        out_tflite_path = os.path.join(TFLITE_EXPORT_DIR, f"{model_tag}_{qmode}.tflite")
        try:
            tflite_bytes = run_tflite_conversion(qmode, allow_flex_ops=False)
            with open(out_tflite_path, "wb") as tf_f:
                tf_f.write(tflite_bytes)
            export_status[f"{qmode}_tflite"] = out_tflite_path
        except Exception:
            try:
                tflite_bytes = run_tflite_conversion(qmode, allow_flex_ops=True)
                with open(out_tflite_path, "wb") as tf_f:
                    tf_f.write(tflite_bytes)
                export_status[f"{qmode}_tflite"] = out_tflite_path
                export_status["ops_fallback_used"].append(f"{qmode}:SELECT_TF_OPS")
            except Exception as flex_err:
                export_status["error"] = str(flex_err)

    return export_status


def locate_yolo_tflite_artifact(exported_return, weights_path):
    if exported_return:
        p = Path(str(exported_return))
        if p.is_file() and p.suffix == ".tflite":
            return p
        if p.is_dir():
            found = sorted(p.rglob("*.tflite"))
            if found:
                return found[0]
    parent_dir = Path(weights_path).parent
    found_parent = sorted(parent_dir.rglob("*.tflite"))
    if found_parent:
        return found_parent[-1]
    return None


EXPORT_REPORT = []

best_m2f_sc = select_best_scenario(BENCHMARK_RECORDS, "Mask2Former-Large")
best_m2f_dir = os.path.join(CHECKPOINT_DIR, f"mask2former_large_scenario_{best_m2f_sc}")
if os.path.exists(best_m2f_dir):
    m2f_model = Mask2FormerForUniversalSegmentation.from_pretrained(best_m2f_dir).cpu().eval()
    m2f_wrapper = ExportableMask2FormerWrapper(m2f_model).eval()
    m2f_dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, dtype=torch.float32)
    m2f_onnx_path = os.path.join(TFLITE_EXPORT_DIR, "mask2former_large_best.onnx")
    try:
        export_to_onnx_compat(
            m2f_wrapper,
            m2f_dummy,
            m2f_onnx_path,
            ["pixel_values"],
            ["class_queries_logits", "masks_queries_logits"],
        )
        m2f_rep = convert_onnx_to_tflite_with_fallback(m2f_onnx_path, "mask2former_large_best")
    except Exception as exc:
        m2f_rep = {"model_tag": "mask2former_large_best", "error": str(exc)}
    EXPORT_REPORT.append(m2f_rep)
    del m2f_model, m2f_wrapper

best_mrcnn_sc = select_best_scenario(BENCHMARK_RECORDS, "Mask-RCNN-Large")
best_mrcnn_ckpt = os.path.join(CHECKPOINT_DIR, f"maskrcnn_large_scenario_{best_mrcnn_sc}_best.pth")
if os.path.exists(best_mrcnn_ckpt):
    mrcnn_model = build_maskrcnn_large(NUM_CLASSES + 1)
    mrcnn_model.load_state_dict(torch.load(best_mrcnn_ckpt, map_location="cpu"))
    mrcnn_model.eval()
    mrcnn_wrapper = ExportableMaskRCNNWrapper(mrcnn_model).eval()
    mrcnn_dummy = torch.rand(1, 3, IMG_SIZE, IMG_SIZE, dtype=torch.float32)
    mrcnn_onnx_path = os.path.join(TFLITE_EXPORT_DIR, "maskrcnn_large_best.onnx")
    try:
        export_to_onnx_compat(
            mrcnn_wrapper,
            mrcnn_dummy,
            mrcnn_onnx_path,
            ["pixel_values"],
            ["boxes", "labels", "scores", "masks"],
        )
        mrcnn_rep = convert_onnx_to_tflite_with_fallback(mrcnn_onnx_path, "maskrcnn_large_best")
    except Exception as exc:
        mrcnn_rep = {"model_tag": "maskrcnn_large_best", "error": str(exc)}
    EXPORT_REPORT.append(mrcnn_rep)
    del mrcnn_model, mrcnn_wrapper

best_yolo_sc = select_best_scenario(BENCHMARK_RECORDS, "YOLOv26l-seg")
best_yolo_ckpt = os.path.join(CHECKPOINT_DIR, f"yolo26l_seg_scenario_{best_yolo_sc}_best.pt")
if os.path.exists(best_yolo_ckpt):
    yolo_best = YOLO(best_yolo_ckpt)
    yolo_rep = {"model_tag": "yolo26l_seg_best", "fp16_tflite": None, "int8_tflite": None, "error": None}
    try:
        exported_fp16 = yolo_best.export(format="tflite", imgsz=IMG_SIZE, half=True)
        fp16_src = locate_yolo_tflite_artifact(exported_fp16, best_yolo_ckpt)
        if fp16_src is not None and fp16_src.exists():
            dest_fp16 = os.path.join(TFLITE_EXPORT_DIR, "yolo26l_seg_best_fp16.tflite")
            shutil.copy2(fp16_src, dest_fp16)
            yolo_rep["fp16_tflite"] = dest_fp16
    except Exception as exc:
        yolo_rep["error"] = str(exc)

    try:
        exported_int8 = yolo_best.export(format="tflite", imgsz=IMG_SIZE, int8=True, data=YOLO_SEG_YAML)
        int8_src = locate_yolo_tflite_artifact(exported_int8, best_yolo_ckpt)
        if int8_src is not None and int8_src.exists():
            dest_int8 = os.path.join(TFLITE_EXPORT_DIR, "yolo26l_seg_best_int8.tflite")
            shutil.copy2(int8_src, dest_int8)
            yolo_rep["int8_tflite"] = dest_int8
    except Exception as exc:
        yolo_rep["error"] = f"{yolo_rep['error']} | INT8: {exc}"
    EXPORT_REPORT.append(yolo_rep)

export_report_path = os.path.join(TFLITE_EXPORT_DIR, "tflite_export_report.json")
with open(export_report_path, "w", encoding="utf-8") as ef:
    json.dump(EXPORT_REPORT, ef, indent=2)

print(json.dumps(EXPORT_REPORT, indent=2))
